# Data Ingestion

In [1]:
import pandas as pd

# 1. Ingestion: Load the CSV from the shared directory
file_path = "manufacturing_defects.csv" # Ensure this matches your exact filename
print(f"Loading data from {file_path}...")
df = pd.read_csv(file_path)

# 2. Basic Validation: Check the shape of the data (rows, columns)
print(f"Dataset successfully loaded. Shape: {df.shape}")

# 3. Quality Check: Look for missing values
missing_data = df.isnull().sum()
print("\nMissing values per column:")
print(missing_data[missing_data > 0]) # Only prints columns that actually have missing data

# 4. Preview the data
display(df.head())

Loading data from manufacturing_defects.csv...
Dataset successfully loaded. Shape: (3240, 17)

Missing values per column:
Series([], dtype: int64)


,ProductionVolume,ProductionCost,SupplierQuality,DeliveryDelay,DefectRate,QualityScore,MaintenanceHours,DowntimePercentage,InventoryTurnover,StockoutRate,WorkerProductivity,SafetyIncidents,EnergyConsumption,EnergyEfficiency,AdditiveProcessTime,AdditiveMaterialCost,DefectStatus
0,202,13175.403783,86.648534,1,3.121492,63.463494,9,0.052343,8.630515,0.081322,85.042379,0,2419.616785,0.468947,5.551639,236.439301,1
1,535,19770.046093,86.310664,4,0.819531,83.697818,20,4.908328,9.296598,0.038486,99.657443,7,3915.566713,0.119485,9.080754,353.957631,1
2,960,19060.820997,82.132472,0,4.514504,90.350550,1,2.464923,5.097486,0.002887,92.819264,2,3392.385362,0.496392,6.562827,396.189402,1
3,370,5647.606037,87.335966,5,0.638524,67.628690,8,4.692476,3.577616,0.055331,96.887013,8,4652.400275,0.183125,8.097496,164.135870,1
4,206,7472.222236,81.989893,3,3.867784,82.728334,9,2.746726,6.851709,0.068047,88.315554,7,1581.630332,0.263507,6.406154,365.708964,1


# Feature Engineering

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separate features (X) and target (y)
X = df.drop('DefectStatus', axis=1)
y = df['DefectStatus']

# 2. Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 3. Scale the numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete.")

Training data shape: (2592, 16)
Testing data shape: (648, 16)
Feature scaling complete.


# Baseline Model Training

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Initialize and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 2. Make predictions on the test set
predictions = model.predict(X_test_scaled)

# 3. Evaluate the model
accuracy = accuracy_score(y_test, predictions)
print(f"Baseline Model Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

Baseline Model Accuracy: 95.52%

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.77      0.84       102
           1       0.96      0.99      0.97       546

    accuracy                           0.96       648
   macro avg       0.94      0.88      0.91       648
weighted avg       0.95      0.96      0.95       648



# MLFLOW

In [9]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Connect to the MLflow tracking server (running in Docker stack)
mlflow.set_tracking_uri("http://mlflow:5000")

# 2. Create or set the experiment
mlflow.set_experiment("Manufacturing_Defects_Prediction")

# 3. Start the tracking run
with mlflow.start_run():
    # Define parameters
    n_estimators = 100
    random_state = 42
    
    # Log parameters to MLflow
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("random_state", random_state)
    
    # Train the model
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
    model.fit(X_train_scaled, y_train)
    
    # Predict and calculate accuracy
    predictions = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, predictions)
    
    # Log the accuracy metric
    mlflow.log_metric("accuracy", accuracy)
    
    # Log the model artifact
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print(f"Run successfully logged to MLflow. Accuracy: {accuracy * 100:.2f}%")

/opt/conda/lib/python3.11/site-packages/_distutils_hack/__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Run successfully logged to MLflow. Accuracy: 95.52%


# Model Validation 

In [11]:
import mlflow
from mlflow.tracking import MlflowClient

# 1. Connect to MLflow
mlflow.set_tracking_uri("http://mlflow:5000")
client = MlflowClient()
experiment = mlflow.get_experiment_by_name("Manufacturing_Defects_Prediction")

# 2. Fetch the latest run
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id], 
    order_by=["start_time DESC"], 
    max_results=1
)
latest_run_id = runs.iloc[0].run_id
latest_accuracy = runs.iloc[0]["metrics.accuracy"]

print(f"Validating Run ID: {latest_run_id}")
print(f"Model Accuracy: {latest_accuracy * 100:.2f}%\n")

# 3. Validate and Register
THRESHOLD = 0.90

if latest_accuracy >= THRESHOLD:
    print(f"Validation Passed! Accuracy is above {THRESHOLD * 100}%.")
    
    # Register the model in MLflow
    model_uri = f"runs:/{latest_run_id}/random_forest_model"
    model_name = "Manufacturing_Defects_RF_Model"
    
    result = mlflow.register_model(model_uri, model_name)
    print(f"Model successfully registered as: {result.name} (Version {result.version})")
    
    # Transition to Staging
    client.transition_model_version_stage(
        name=model_name,
        version=result.version,
        stage="Staging"
    )
    print("Model transitioned to Staging.")
else:
    print("Validation Failed. Model accuracy is too low for deployment.")

Validating Run ID: 123aba162ed8452896e67cf4d03bd8df
Model Accuracy: 95.52%

Validation Passed! Accuracy is above 90.0%.


Registered model 'Manufacturing_Defects_RF_Model' already exists. Creating a new version of this model...
2026/09/30 10:45:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Manufacturing_Defects_RF_Model, version 2
Created version '2' of model 'Manufacturing_Defects_RF_Model'.
/tmp/ipykernel_231/226585013.py:35: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model successfully registered as: Manufacturing_Defects_RF_Model (Version 2)
Model transitioned to Staging.
